## Analyze the impact of optimization, weight Initialization and regularization


### 1. Experimenting with Optimization Algorithms on CIFAR-10

##### Import Necessary Libraries

In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers  import Dense, Flatten
from tensorflow.keras.optimizers import  SGD, RMSprop, Adagrad, Adam
import matplotlib.pyplot as plt



#### plot learning curves- function

In [ ]:
import matplotlib.pyplot as plt

def plot_training_history(history, header):
    """
    Plots training and validation accuracy and loss across epochs, with a header based on the options.

    Args:
        history: Training history object returned by the model.fit() method.
        header: String specifying the option.
    """
    plt.figure(figsize=(12, 4))

    # Plot accuracy
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title(f'Accuracy {header}')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    # Plot loss
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title(f'Loss {header}')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.show()


##### Load and Preprocess Data

In [ ]:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Normalize pixel values to be between 0 and 1
x_train, x_test = x_train / 255.0, x_test / 255.0

# Convert class vectors to binary class matrices

num_classes = 10
y_train = tf.keras.utils.to_categorical(y_train, num_classes)
y_test = tf.keras.utils.to_categorical(y_test, num_classes)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


##### Define the Model

In [ ]:
  def create_model():
    model = Sequential([
      Flatten(input_shape=(32, 32, 3)),
      Dense(512, activation='relu'),
      Dense(256, activation='relu'),
      Dense(128, activation='relu'),
      Dense(num_classes, activation='softmax')
    ])
    return model

##### Define Optimization Algorithms and Hyperparameters

In [ ]:
optimizers = {
    'GD': SGD(learning_rate=0.01),
    'SGD': SGD(learning_rate=0.01, momentum=0.9),
    'MiniBatch': SGD(learning_rate=0.01),  # Mini-batch is handled by batch_size parameter
    'RMSprop': RMSprop(learning_rate=0.001),
    'Adagrad': Adagrad(learning_rate=0.01),
    'Adam': Adam(learning_rate=0.001)
}

epochs = 10
batch_size = 32

##### Training and Evaluation Loop

In [ ]:
def train_and_evaluate(optimizer):
  model = create_model()
  model.compile(loss='categorical_crossentropy',
                optimizer=optimizer,
                metrics=['accuracy'])

  history = model.fit(x_train, y_train,
                    batch_size=batch_size,
                    epochs=epochs,
                    validation_data=(x_test, y_test))

  # Plot learning curves here using your plot_training_history function
  # plot_training_history(history,'(optimizer: {optimizer}' )

  test_loss, test_acc = model.evaluate(x_test, y_test)
  print('Test accuracy:', test_acc)
  return test_acc, history

##### Iterate Over Optimizers

In [ ]:
# @title
results = {}
for optimizer_name, optimizer in optimizers.items():
  print(f"Training with optimizer: {optimizer_name}")
  accuracy, history = train_and_evaluate(optimizer)
  results[optimizer_name] = accuracy


Training with optimizer: GD


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.2785 - loss: 1.9863 - val_accuracy: 0.3745 - val_loss: 1.7457
Epoch 2/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.3962 - loss: 1.6956 - val_accuracy: 0.4362 - val_loss: 1.5895
Epoch 3/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.4311 - loss: 1.6025 - val_accuracy: 0.4240 - val_loss: 1.5851
Epoch 4/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.4536 - loss: 1.5436 - val_accuracy: 0.4465 - val_loss: 1.5605
Epoch 5/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.4673 - loss: 1.4958 - val_accuracy: 0.4815 - val_loss: 1.4564
Epoch 6/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.4871 - loss: 1.4416 - val_accuracy: 0.4564 - val_loss: 1.4954
Epoch 7/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.4979 - loss: 1.4033 - val_accuracy: 0.4925 - val_loss: 1.4320
Epoch 8/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.5134 - loss: 1.3744 - 

##### Compare results

In [ ]:
print(results)

{'GD': 0.4823000133037567, 'SGD': 0.4778999984264374, 'MiniBatch': 0.5095999836921692, 'RMSprop': 0.43309998512268066, 'Adagrad': 0.49709999561309814, 'Adam': 0.47749999165534973}


## 2. Experimenting with Initialization Techniques on CIFAR-10
This code explores the impact of different initialization techniques (Xavier and He normalization) on a feedforward neural network for CIFAR-10 classification in Keras.

##### Imports

In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.initializers  import glorot_uniform, he_normal

import matplotlib.pyplot as plt

##### Load and Preprocess Data

In [ ]:

# Load CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Normalize pixel values to    range [0, 1]
x_train, x_test = x_train / 255.0, x_test / 255.0

# Convert class vectors to binary class matrices
num_classes = 10
y_train = tf.keras.utils.to_categorical(y_train, num_classes)
y_test = tf.keras.utils.to_categorical(y_test, num_classes)

##### Define the Model

In [ ]:
def create_model(init_type):
  model = Sequential([
    Flatten(input_shape=(32, 32, 3)),
    Dense(512, activation='relu', kernel_initializer=init_type),
    Dense(256, activation='relu', kernel_initializer=init_type),
    Dense(128, activation='relu', kernel_initializer=init_type),
    Dense(num_classes, activation='softmax')
  ])
  return model

##### Define Training and Evaluation Function

In [ ]:
def train_and_evaluate(init_type):
  model = create_model(init_type)
  model.compile(loss='categorical_crossentropy',
                optimizer='adam',
                metrics=['accuracy'])

  history = model.fit(x_train, y_train,
                    epochs=10,
                    batch_size=32,
                    validation_data=(x_test, y_test))


  # plot_training_history(history, '(Initialization: {init_type}')  # Call your learning curve plotting function

  test_loss, test_acc = model.evaluate(x_test, y_test)
  print('Test accuracy:', test_acc)

  return test_acc

##### Experiment with Initializers

In [ ]:
initializers = {
  'Xavier': glorot_uniform,
  'HeNormal': he_normal
}

results = {}
for init_name, init_func in initializers.items():
  print(f"Training with initialization: {init_name}")
  accuracy = train_and_evaluate(init_func)
  results[init_name] = accuracy

Training with initialization: Xavier
Epoch 1/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.2743 - loss: 1.9957 - val_accuracy: 0.3885 - val_loss: 1.7023
Epoch 2/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.3834 - loss: 1.7099 - val_accuracy: 0.4381 - val_loss: 1.6030
Epoch 3/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.4214 - loss: 1.6102 - val_accuracy: 0.4427 - val_loss: 1.5598
Epoch 4/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.4505 - loss: 1.5381 - val_accuracy: 0.4334 - val_loss: 1.5668
Epoch 5/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.4614 - loss: 1.5052 - val_accuracy: 0.4584 - val_loss: 1.5155
Epoch 6/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.4756 - loss: 1.4650 - val_accuracy: 0.4665 - val_loss: 1.4996
Epoch 7/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - accuracy: 0.4841 - loss: 1.4375 - val_accuracy: 0.4799 - val_loss: 1.4704
Epoch 8/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/ste

##### Compare Results

In [ ]:
print(results)

{'Xavier': 0.48190000653266907, 'HeNormal': 0.4814000129699707}


## 3. Experimenting with Regularization Techniques on CIFAR-10
This code explores the effects of different regularization techniques (L2 weight decay, dropout, and early stopping) on a feedforward neural network for CIFAR-10 image classification in Keras.

##### Imports

In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers  import Dense, Flatten, Dropout
from tensorflow.keras.callbacks import EarlyStopping

import matplotlib.pyplot as plt

##### Load and Preprocess Data

In [ ]:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Normalize pixel values to be between 0 and 1
x_train, x_test = x_train / 255.0, x_test / 255.0

# Convert class vectors to binary class matrices

num_classes = 10
y_train = tf.keras.utils.to_categorical(y_train, num_classes)
y_test = tf.keras.utils.to_categorical(y_test, num_classes)

##### Define the Model

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten

def create_model(regularization_type, l2_reg=0.0):
    model = Sequential()
    model.add(Flatten(input_shape=(32, 32, 3)))

    # First Dense layer
    model.add(Dense(512, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(l2_reg) if regularization_type == 'L2' else None))
    if regularization_type == 'Dropout':
        model.add(Dropout(0.2))

    # Second Dense layer
    model.add(Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(l2_reg) if regularization_type == 'L2' else None))
    if regularization_type == 'Dropout':
        model.add(Dropout(0.2))

    # Third Dense layer
    model.add(Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(l2_reg) if regularization_type == 'L2' else None))

    # Output layer
    model.add(Dense(num_classes, activation='softmax'))

    return model


##### Define Training and Evaluation Function

In [ ]:
def train_and_evaluate(regularization_type, l2_reg=0.0, patience=5):
  model = create_model(regularization_type, l2_reg)
  model.compile(loss='categorical_crossentropy',
                optimizer='adam',
                metrics=['accuracy'])

  early_stopping = EarlyStopping(monitor='val_accuracy', patience=patience)  # Early stopping callback

  history = model.fit(x_train, y_train,
                    epochs=20,  # May need adjustments based on stopping criteria
                    batch_size=32,
                    validation_data=(x_test, y_test),
                    callbacks=[early_stopping])

  # plot_training_history(history, regularization_type)  # Call your learning curve plotting function

  test_loss, test_acc = model.evaluate(x_test, y_test)
  print('Test accuracy:', test_acc)

  return test_acc

##### Experiment with Regularization Techniques

In [ ]:
regularization_options = {
    'Baseline': None,  # No regularization
    'L2': 0.001,
    'Dropout': None,  # Dropout is applied in the model creation function
    'EarlyStopping': None  # Patience defined in the training function
}

results = {}
for reg_name, reg_param in regularization_options.items():
    print(f"Training with regularization: {reg_name}")
    if reg_name == 'EarlyStopping':
        accuracy = train_and_evaluate(reg_name, patience=10)  # Experiment with patience value
    else:
        accuracy = train_and_evaluate(reg_name, reg_param)
    results[reg_name] = accuracy


Training with regularization: Baseline
Epoch 1/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.2714 - loss: 1.9970 - val_accuracy: 0.3821 - val_loss: 1.7272
Epoch 2/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.3907 - loss: 1.6958 - val_accuracy: 0.4393 - val_loss: 1.5722
Epoch 3/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.4246 - loss: 1.6050 - val_accuracy: 0.4412 - val_loss: 1.5736
Epoch 4/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.4475 - loss: 1.5430 - val_accuracy: 0.4622 - val_loss: 1.5064
Epoch 5/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - accuracy: 0.4596 - loss: 1.5053 - val_accuracy: 0.4744 - val_loss: 1.4901
Epoch 6/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.4766 - loss: 1.4492 - val_accuracy: 0.4765 - val_loss: 1.4739
Epoch 7/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.4849 - loss: 1.4320 - val_accuracy: 0.4667 - val_loss: 1.5149
Epoch 8/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/s

##### Compare Results

In [ ]:
print(results)

{'Baseline': 0.4982999861240387, 'L2': 0.4652000069618225, 'Dropout': 0.42989999055862427, 'EarlyStopping': 0.48919999599456787}
